In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("ggseqlogo", "ggplot2")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)

[[1]]
[1] "ggseqlogo" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[7] "datasets"  "methods"   "base"     

[[2]]
 [1] "ggplot2"   "ggseqlogo" "repr"      "stats"     "graphics"  "grDevices"
 [7] "utils"     "datasets"  "methods"   "base"

In [2]:
# load data
gstar_files <- dir(path = "./lib/AllenScore/", pattern = "\\.PPR.txt")
gstar_files <- paste0("./lib/AllenScore/", gstar_files)
ppr_list <- read.delim2("./lib/conserved_sirna_ppr_accs.txt", sep = "\t", header = FALSE)[[1]]
id_map <- read.delim2("./lib/col_accs_geneid.txt", sep = "\t", header = TRUE)


In [3]:
temp_list <- lapply(gstar_files, function(f) {

  acc <- sub("\\..*", "", f)

  gstar <- read.delim2(f, sep = "\t", header = TRUE,
                       comment.char = "#")

  finfo <- gstar[, c("Query", "Transcript", "AllenScore", "Sequence")]
  finfo <- finfo[order(finfo$AllenScore), ]

  finfo$ids  <- finfo$Transcript
  finfo$gene <- finfo$Transcript

  finfo$target_seq <- gsub("-", "", sub("&.*$", "", finfo$Sequence))
  finfo$sRNA_seq   <- gsub("-", "", sub("^.*?&", "", finfo$Sequence))

  finfo$target_seq_len <- nchar(finfo$target_seq)
  finfo$sRNA_seq_len   <- nchar(finfo$sRNA_seq)

  finfo$acc <- acc
  finfo
})

temp <- do.call(rbind, temp_list)

In [4]:
query_mapping <- c(
  "TAS2_3D6"   = "TAS2 3'D6",
  "miR161.1"   = "miR161.1",
  "miR161.2"   = "miR161.2",
  "miR400"     = "miR400",
  "TAS1a_3D9"  = "TAS1a 3'D9",
  "TAS1b_3D4"  = "TAS1b 3'D4",
  "TAS1c_3D6"  = "TAS1c 3'D6",
  "TAS1c_3D10" = "TAS1c 3'D10",
  "TAS2_3D9"   = "TAS2 3'D9",
  "TAS2_3D11"  = "TAS2 3'D11",
  "TAS2_3D12"  = "TAS2 3'D12"
)

idx <- match(temp$Query, names(query_mapping))
temp$Query[!is.na(idx)] <- query_mapping[idx[!is.na(idx)]]


In [5]:
temp$AllenScore <- as.numeric(temp$AllenScore)
temp$gene <- id_map$Col.0_ID[ match(temp$ids, id_map$Accession_ID)]

temp <- temp[temp$ids %in% ppr_list, ]
temp <- temp[temp$AllenScore < 4, ]
temp <- unique(temp)

In [6]:
unisRNA <- unique(temp$Query)

for (q in unisRNA) {
  
  pdata <- temp[temp$Query == q, ]
  ntars <- nrow(pdata)
  
  tarseq <- pdata$target_seq[pdata$target_seq_len == 21]
  
  if (length(tarseq) == 0) next
  
  title <- sprintf("%s targets: n = %d", q, ntars)
  message(title)
  
  p <- ggplot() +
    geom_logo(tarseq) +
    theme_logo() +
    theme_classic() +
    scale_x_continuous(breaks = seq(0, 21, 3), expand = c(0.01, 0)) +
    ggtitle(title)
  
  ggsave(p, file = paste0(q, "_target.pdf"), height = 1.2, width  = 4.3 )
}


TAS2 3'D6 targets: n = 59

Warning message:
“The `<scale>` argument of `guides()` cannot be `FALSE`. Use "none" instead as of ggplot2 3.3.4.
ℹ The deprecated feature was likely used in the ggseqlogo package.
  Please report the issue at <https://github.com/omarwagih/ggseqlogo/issues>.”
Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.
ℹ The deprecated feature was likely used in the ggseqlogo package.
  Please report the issue at <https://github.com/omarwagih/ggseqlogo/issues>.”
Scale for x is already present.
Adding another scale for x, which will replace the existing scale.
miR161.2 targets: n = 51

Scale for x is already present.
Adding another scale for x, which will replace the existing scale.
TAS1c 3'D10 targets: n = 32

Scale for x is already present.
Adding another scale for x, which will replace the existing scale.
TAS1a 3'D9 targets: n = 41

Scal

In [7]:
sessionInfo()

R version 4.5.1 (2025-06-13)
Platform: aarch64-apple-darwin23.6.0
Running under: macOS Sonoma 14.5

Matrix products: default
BLAS:   /opt/homebrew/Cellar/openblas/0.3.30/lib/libopenblasp-r0.3.30.dylib 
LAPACK: /opt/homebrew/Cellar/r/4.5.1/lib/R/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] en_GB.UTF-8/en_GB.UTF-8/en_GB.UTF-8/C/en_GB.UTF-8/en_GB.UTF-8

time zone: America/Chicago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] ggplot2_4.0.1   ggseqlogo_0.2.2 repr_1.1.7     

loaded via a namespace (and not attached):
 [1] crayon_1.5.3       vctrs_0.6.5        cli_3.6.5          rlang_1.1.6       
 [5] generics_0.1.4     textshaping_1.0.4  S7_0.2.1           jsonlite_2.0.0    
 [9] labeling_0.4.3     glue_1.8.0         htmltools_0.5.9    IRdisplay_1.1     
[13] IRkernel_1.3.2     ragg_1.5.0         scales_1.4.0       grid_4.5.1        
[17] tibble_3.3.0       evaluate_1.0.5